# 01 — EDA מאוחד: SHADY

מחברת זו מאחדת את ניתוח הנתונים של כל שכבות הפרויקט:

| קטע | שכבה | מקור |
|-----|------|------|
| 1 | מבנים | עיריית ת"א — `buildings_clean.csv` |
| 2 | עצים | מפ"י — `national_canopy_clean.parquet` |
| 3 | זווית שמש | PySolar — חישוב לפי שעה ועונה |
| 4 | מזג אוויר | Open-Meteo API + fallback חודשי |

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import json

DATA_DIR = Path("../data")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"
print("Setup OK")

---
## קטע 1 — מבנים
**מקור:** `data/buildings_clean.csv` — 45,783 מבנים, עיריית תל אביב.  
**עמודות מרכזיות:** `height` (מ'), `height_source` (recorded / imputed), `floors`, `building_type`.

In [ ]:
buildings = pd.read_csv(DATA_DIR / "buildings_clean.csv")
print(f"שורות: {len(buildings):,}  |  עמודות: {buildings.shape[1]}")
buildings.head(3)

In [ ]:
# KPIs
print(f"מבנים בסה""כ:        {len(buildings):,}")
print(f"גובה ממוצע:          {buildings['height'].mean():.1f} מ'")
print(f"גובה חציוני:         {buildings['height'].median():.1f} מ'")
print(f"מבנים > 20 מ':       {(buildings['height'] > 20).sum():,}")
print(f"גובה מחושב (imputed): {(buildings['height_source'] == 'imputed').mean() * 100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# גרף 1 — היסטוגרמת גבהים
ax = axes[0]
d_rec = buildings.loc[buildings["height_source"] == "recorded", "height"].clip(upper=60)
d_imp = buildings.loc[buildings["height_source"] == "imputed",  "height"].clip(upper=60)
ax.hist(d_rec, bins=40, color="#2980b9", alpha=0.8, label=f"recorded ({len(d_rec):,})")
ax.hist(d_imp, bins=40, color="#e67e22", alpha=0.8, label=f"imputed  ({len(d_imp):,})")
ax.set_xlabel("Height (m)")
ax.set_ylabel("Count")
ax.set_title("Building Height Distribution (clipped at 60 m)")
ax.legend()

# גרף 2 — קומות מול גובה
ax2 = axes[1]
pool = buildings[buildings["floors"].notna() & buildings["floors"].between(1, 25)]
samp = pool.sample(min(2000, len(pool)), random_state=42)
colors = samp["height_source"].map({"recorded": "#2980b9", "imputed": "#e67e22"})
ax2.scatter(samp["floors"], samp["height"], c=colors, alpha=0.25, s=8)
x = np.linspace(1, 25, 100)
ax2.plot(x, 2.95 * x + 5.21, color="red", lw=2, label="y = 2.95x + 5.21  (r=0.70)")
ax2.set_xlabel("Floors (ms_komot)")
ax2.set_ylabel("Height (m)")
ax2.set_title("Floors vs. Height")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# גרף 3 — סוגי מבנים (Top 8)
type_counts = buildings["building_type"].value_counts().head(8)
fig3, ax3 = plt.subplots(figsize=(11, 3))
bars = ax3.barh(type_counts.index[::-1], type_counts.values[::-1], color="#2980b9")
for bar, val in zip(bars, type_counts.values[::-1]):
    ax3.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
             f"{val:,}", va="center", fontsize=9)
ax3.set_xlabel("Count")
ax3.set_title("Building Types (Top 8)")
ax3.set_xlim(0, type_counts.max() * 1.12)
plt.tight_layout()
plt.show()

---
## קטע 2 — עצים
**מקור:** `data/national_canopy_clean.parquet` — 231,234 פוליגוני חופת עצים, מפ"י.  
**עמודות מרכזיות:** `canopy_area_m2`, `canopy_perimeter_m`, `lat`, `lon`, `area_class`.

In [ ]:
trees = pd.read_parquet(DATA_DIR / "national_canopy_clean.parquet",
                        columns=["OBJECTID", "canopy_perimeter_m", "canopy_area_m2",
                                 "lon", "lat", "Geometry_Type", "area_class"])
print(f"שורות: {len(trees):,}  |  NaN בסה""כ: {trees.isna().sum().sum()}")
trees.describe().T.round(2)

In [ ]:
# גרף 1 — התפלגות canopy_area_m2 (log Y) + bar chart קטגוריות
area_class = trees["area_class"].str.replace("\ufffd", "\u00b2", regex=False)

fig, (ax_h, ax_b) = plt.subplots(1, 2, figsize=(13, 4))

# היסטוגרמה
bins = np.arange(0, 205, 5)
ax_h.hist(trees["canopy_area_m2"], bins=bins,
          color="#27ae60", alpha=0.85, log=True, edgecolor="white", linewidth=0.5)
ax_h.set_xlabel("Canopy Area (m\u00b2)")
ax_h.set_ylabel("Count (log scale)")
ax_h.set_title("Distribution of canopy_area_m\u00b2 (clipped at 200)")
ax_h.set_xlim(0, 200)

# bar chart קטגוריות
order = ["0-1 m\u00b2", "1-5 m\u00b2", "5-10 m\u00b2", "10-25 m\u00b2",
         "25-50 m\u00b2", "50-100 m\u00b2", "100+ m\u00b2"]
ac = area_class.value_counts().reindex([c for c in order if c in area_class.value_counts()])
ax_b.bar(range(len(ac)), ac.values, color="#16a085")
for i, v in enumerate(ac.values):
    ax_b.text(i, v + 500, f"{v:,}", ha="center", fontsize=8)
ax_b.set_xticks(range(len(ac)))
ax_b.set_xticklabels(ac.index, rotation=30, ha="right", fontsize=9)
ax_b.set_ylabel("Count")
ax_b.set_title("Canopy size categories")

plt.tight_layout()
plt.show()

In [ ]:
# גרף 2 — Boxplots (log scale)
fig_bp, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, col, color in [
    (ax1, "canopy_perimeter_m", "#a9dfbf"),
    (ax2, "canopy_area_m2",     "#82e0aa"),
]:
    ax.boxplot(trees[col], vert=False, patch_artist=True,
               boxprops=dict(facecolor=color))
    ax.set_xscale("log")
    ax.set_title(f"{col} (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# גרף 3 — Correlation heatmap
numeric_cols = ["canopy_perimeter_m", "canopy_area_m2", "lat", "lon"]
corr = trees[numeric_cols].corr().round(3)

fig_c, ax_c = plt.subplots(figsize=(6, 5))
im = ax_c.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax_c.set_xticks(range(len(corr.columns)))
ax_c.set_yticks(range(len(corr.columns)))
ax_c.set_xticklabels(corr.columns, rotation=30, ha="right")
ax_c.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax_c.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                  color="white" if abs(corr.values[i, j]) > 0.5 else "black", fontsize=10)
fig_c.colorbar(im, ax=ax_c, shrink=0.7)
ax_c.set_title("Pearson correlation — numeric features")
plt.tight_layout()
plt.show()
print("תובנה: canopy_perimeter_m ו-canopy_area_m2 מקושרים ב-0.96 — נשמור רק area במודל.")

In [ ]:
# גרף 4 — Hexbin פיזור גיאוגרפי
fig_hex, ax_hex = plt.subplots(figsize=(9, 8))
hb = ax_hex.hexbin(
    trees["lon"], trees["lat"],
    C=trees["canopy_area_m2"],
    reduce_C_function=np.mean,
    gridsize=45, cmap="Greens", mincnt=3,
    linewidths=0.1, edgecolors="white",
)
cb = fig_hex.colorbar(hb, ax=ax_hex, shrink=0.7)
cb.set_label("Mean canopy area per hex (m\u00b2)")
ax_hex.set_xlabel("Longitude")
ax_hex.set_ylabel("Latitude")
ax_hex.set_title("Tel Aviv — Where are the big trees?")
ax_hex.set_aspect(1 / np.cos(np.radians(32.08)))
plt.tight_layout()
plt.show()

---
## קטע 3 — זווית שמש (PySolar)
חישוב `altitude` (גובה השמש מעל האופק) ו-`azimuth` (כיוון האופק) לכל שעה,  
עבור יום קיצי (21 יוני) ויום חורפי (21 דצמבר) בת"א.

In [ ]:
try:
    import pysolar.solar as solar
    _PYSOLAR = True
    print("pysolar OK")
except ImportError:
    _PYSOLAR = False
    print("pysolar לא מותקן — pip install pysolar")

In [ ]:
TA_LAT, TA_LON = 32.08, 34.77
HOURS = list(range(5, 21))  # 05:00–20:00

def sun_profile(year, month, day):
    """מחזיר DataFrame עם altitude ו-azimuth לכל שעה ביום נתון."""
    rows = []
    for h in HOURS:
        dt = datetime(year, month, day, h, 0, 0, tzinfo=timezone.utc)
        alt = solar.get_altitude(TA_LAT, TA_LON, dt)
        az  = solar.get_azimuth(TA_LAT, TA_LON, dt)
        rows.append({"hour": h, "altitude": alt, "azimuth": az})
    return pd.DataFrame(rows)

if _PYSOLAR:
    summer = sun_profile(2025, 6, 21)   # יום קיצי
    winter = sun_profile(2025, 12, 21)  # יום חורפי
    print(summer.to_string(index=False))
else:
    print("דלג — pysolar לא מותקן")

In [ ]:
if _PYSOLAR:
    fig, (ax_alt, ax_az) = plt.subplots(1, 2, figsize=(13, 4))

    # גובה שמש
    ax_alt.plot(summer["hour"], summer["altitude"], color="#e74c3c",
                lw=2.5, marker="o", markersize=5, label="21 יוני (קיץ)")
    ax_alt.plot(winter["hour"], winter["altitude"], color="#2980b9",
                lw=2.5, marker="s", markersize=5, label="21 דצמבר (חורף)")
    ax_alt.axhline(0,  color="gray", lw=1, ls="--", label="אופק")
    ax_alt.axhline(20, color="#f39c12", lw=1, ls=":", label="צל פוטנציאלי (<20°)")
    ax_alt.fill_between(summer["hour"], 0, summer["altitude"],
                        where=summer["altitude"] > 0,
                        color="#e74c3c", alpha=0.08)
    ax_alt.set_xlabel("שעה (UTC)")
    ax_alt.set_ylabel("גובה השמש (מעלות)")
    ax_alt.set_title("גובה השמש לאורך היום — קיץ vs. חורף")
    ax_alt.legend(fontsize=9)
    ax_alt.set_xticks(HOURS)

    # אזימוט
    ax_az.plot(summer["hour"], summer["azimuth"], color="#e74c3c",
               lw=2.5, marker="o", markersize=5, label="21 יוני")
    ax_az.plot(winter["hour"], winter["azimuth"], color="#2980b9",
               lw=2.5, marker="s", markersize=5, label="21 דצמבר")
    ax_az.set_xlabel("שעה (UTC)")
    ax_az.set_ylabel("אזימוט (מעלות, 0=צפון)")
    ax_az.set_title("כיוון השמש לאורך היום")
    ax_az.legend(fontsize=9)
    ax_az.set_xticks(HOURS)

    plt.tight_layout()
    plt.show()

    low_alt_hours = summer[summer["altitude"].between(0, 20)]["hour"].tolist()
    print(f"שעות עם altitude < 20° (קיץ): {low_alt_hours}")
    print("תובנה: בשעות אלו צל מבנים ועצים נופח קדימה — המסלול קריטי.")

---
## קטע 4 — מזג אוויר (Open-Meteo)
קריאה ל-Open-Meteo Historical API לשנת 2024 — ממוצעי טמפ', לחות וענן לפי חודש.  
התוצאה נשמרת ב-`data/climate_fallback.json` לשימוש כ-fallback באפליקציה.

In [ ]:
ARCHIVE_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=32.08&longitude=34.77"
    "&start_date=2024-01-01&end_date=2024-12-31"
    "&hourly=temperature_2m,relative_humidity_2m,cloud_cover"
    "&timezone=Asia%2FJerusalem"
)

try:
    resp = requests.get(ARCHIVE_URL, timeout=30)
    resp.raise_for_status()
    raw = resp.json()["hourly"]
    weather_df = pd.DataFrame({
        "time":        pd.to_datetime(raw["time"]),
        "temperature": raw["temperature_2m"],
        "humidity":    raw["relative_humidity_2m"],
        "cloud_cover": raw["cloud_cover"],
    })
    weather_df["month"] = weather_df["time"].dt.month
    print(f"שורות שהתקבלו: {len(weather_df):,}")
    _API_OK = True
except Exception as e:
    print(f"API לא זמין: {e}")
    print("טוען fallback מ-data/climate_fallback.json")
    fb = json.loads((DATA_DIR / "climate_fallback.json").read_text(encoding="utf-8"))
    weather_df = pd.DataFrame(fb)
    weather_df["month"] = weather_df["month"]
    _API_OK = False

In [ ]:
monthly = weather_df.groupby("month")[["temperature", "humidity", "cloud_cover"]].mean().round(1)
monthly.index = [
    "ינואר", "פברואר", "מרץ", "אפריל", "מאי", "יוני",
    "יולי", "אוגוסט", "ספטמבר", "אוקטובר", "נובמבר", "דצמבר"
]
monthly

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#e74c3c", "#3498db", "#95a5a6"]
titles = ["טמפרטורה ממוצעת (°C)", "לחות ממוצעת (%)", "כיסוי ענן ממוצע (%)"]
cols   = ["temperature", "humidity", "cloud_cover"]

for ax, col, color, title in zip(axes, cols, colors, titles):
    bars = ax.bar(range(12), monthly[col], color=color, alpha=0.85)
    ax.set_xticks(range(12))
    ax.set_xticklabels(monthly.index, rotation=45, ha="right", fontsize=8)
    ax.set_title(title)
    for i, v in enumerate(monthly[col]):
        ax.text(i, v + 0.3, f"{v:.0f}", ha="center", fontsize=7)

plt.suptitle("ממוצעי מזג אוויר חודשיים — תל אביב 2024", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# שמירת fallback מעודכן רק אם קיבלנו נתונים מה-API
if _API_OK:
    MONTH_NAMES = [
        "ינואר", "פברואר", "מרץ", "אפריל", "מאי", "יוני",
        "יולי", "אוגוסט", "ספטמבר", "אוקטובר", "נובמבר", "דצמבר"
    ]
    fallback = [
        {
            "month": int(i + 1),
            "month_name": MONTH_NAMES[i],
            "temperature": float(monthly["temperature"].iloc[i]),
            "humidity":    int(monthly["humidity"].iloc[i]),
            "cloud_cover": int(monthly["cloud_cover"].iloc[i]),
        }
        for i in range(12)
    ]
    out_path = DATA_DIR / "climate_fallback.json"
    out_path.write_text(json.dumps(fallback, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved: {out_path}")
else:
    print("API לא זמין — fallback קיים לא עודכן")